# Chapitre 10 - Cours 1 : AdaBoost, modélisation additive pas à pas et perte exponentielle

Ce premier notebook couvre les sections 10.1 à 10.5 du livre :

1. l'idée du *boosting* et l'algorithme **AdaBoost.M1** (algorithme 10.1) ;
2. son efficacité spectaculaire sur la simulation des « sphères emboîtées » (éq. (10.2), figure 10.2) ;
3. la **modélisation additive pas à pas** (*forward stagewise*, algorithme 10.2) ;
4. l'équivalence entre AdaBoost et la modélisation pas à pas avec la **perte exponentielle** (éq. (10.8)–(10.15)) ;
5. ce que la perte exponentielle estime : la moitié du **log-rapport de cotes** (éq. (10.16)–(10.17)).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl.ch10_boosting import AdaBoostM1, RegressionTree, simulate_nested_spheres

## 1. L'idée du boosting

On se place en **classification binaire** : la réponse est codée $Y \in \{-1, +1\}$ et un
classifieur est une fonction $G(x) \in \{-1,+1\}$. Son taux d'erreur sur l'échantillon
d'apprentissage $(x_i, y_i)_{1\le i\le N}$ est

$$\overline{\mathrm{err}} = \frac{1}{N}\sum_{i=1}^N I\big(y_i \neq G(x_i)\big),$$

où $I(\cdot)$ vaut 1 si la condition est vraie et 0 sinon.

Un **classifieur faible** est un classifieur à peine meilleur que le hasard (erreur
légèrement inférieure à $1/2$). Le boosting applique *séquentiellement* un même
algorithme faible à des versions **repondérées** des données, produisant des
classifieurs $G_1, \dots, G_M$, puis les combine par un **vote majoritaire pondéré** :

$$G(x) = \operatorname{signe}\Big(\sum_{m=1}^M \alpha_m G_m(x)\Big) \qquad \text{(éq. (10.1) du livre).}$$

> **Intuition.** À chaque étape, on augmente le poids des observations que le classifieur
> précédent a mal classées : le classifieur suivant est ainsi « forcé » de se concentrer
> sur les cas difficiles. Les coefficients $\alpha_m$ donnent plus d'influence aux
> classifieurs les plus précis.